# 11. Time series tool for analysing HDF5/pandas files 

### Setup notebook

In [ ]:
# Reload code outside notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook
%matplotlib notebook

### Imports

In [ ]:
import os
import glob
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from zipfile import ZipFile

# PlatoSim
import platosim.plot as pt
import platosim.utilities as ut
import platosim.referenceFrames as rf
from platosim.photometryfile import PhotometryFile
from platosim.matplotlibrc import setup
setup()

### Run a default simulation for the tutorial

In [ ]:
outputDir      = os.getcwd()
outputFile     = f"{outputDir}/{outputFileName}.hdf5"

---

##  Basic overview

We have alread touched upon how you can use the `SimFile` class to retrieve information from the HDF5 file. Here we dive in a bit deeper and show several functionalites. We will look at the following APIs:

In [ ]:
# Read path
path = '/lhome/nicholas/sims_kul21/jitterNone/000000001/' 

# Fetch all zip files
files = glob.glob(path+ "*.zip")

# Unpack zip files
with ZipFile(files[0], 'r') as unzip:
    unzip.extractall(path)
    
# Get file names
filename_ftr = files[0][:-3] + "ftr"
filename_cat = files[0][:-3] + "cat"

let's start by a simple example of extracting the simulated light curve and plotting a 1 hour binned curve on top which is the general time scale used in the PLATO mission for noise subpression of finding exoplanets.

In [ ]:
# Load feather file
lc = PhotometryFile(filename_ftr)

In [ ]:
# Get obs information; [group, camera, quarter] 
lc.obs()

In [ ]:
# Create a plot of 
fig, ax = lc.plot(flux_unit="e/s", errorbar=False, median_filter=True, binsize=1)
ax.set_title('Light curve example');

It is straight away to fetch the rebinned light curve shown above a red markers:

In [ ]:
time_bin, flux_bin, tdur_bin = lc.bin(unit="ppm")
flux_bin

## Photometric precision

To analyse the noise level in a ligth curve a few options are available. The first is the Noise-to-Signal Ratio (NSR) defined by:

$$
{\rm NSR}_{\rm 1h} = \frac{10^6}{\sqrt{144}}\frac{\sigma_F}{\bar{F}}
$$

In [ ]:
lc.getNSR()

It is easy to get the Root-Mean-Square of the noise level in a light curve defined by

$$
{\rm RMS}_{\rm 1h} = \frac{10^6}{\sqrt{144}}\frac{F_{\rm median}}{\bar{F}}
$$

In [ ]:
# Fetch NSR [ppm/sqrt(h)]
lc.getRMS(column="flux_med", unit="ppm")

Lastly the Median Absolute Deviation (MAD) can also be fetched by:

In [ ]:
lc.getMAD(column="flux_med", unit="ppm")

In [ ]:
# Mean flux error in percent
flux     = lc.flux()
flux_err = lc.flux_err()
flux_err_percent = flux_err.mean() / flux.mean() * 100
flux_err_percent

In [ ]:
# Mean centroid error in percent
xcen = lc.xcen(unit="pix")
ycen = lc.ycen(unit="pix")
xcen_err = lc.xcen_err(unit="pix")
ycen_err = lc.ycen_err(unit="pix")
rcen     = np.sqrt(xcen**2 + ycen**2)
rcen_err = np.sqrt(xcen_err**2 + ycen_err**2)
rcen_err_percent = rcen_err.mean() / rcen.mean() * 100
rcen_err_percent

In [ ]:
lc.time()
# phot.flux_err()
# xcen =phot.xcen(unit="rel")
# phot.xcen_err(unit="pix")
# ycen = phot.ycen(unit="rel")
# phot.xcen_err(unit="pix")

### Centroid and jitter

In [ ]:
lc.plotCentroid(cen_unit="rel");


In [ ]:
xcen = dc["x"][0] - 3.5
ycen = dc["y"][0] - 3.5
rcen = np.sqrt(xcen**2 + ycen**2)
rcen

---

## Multi-camera photometry

In [ ]:
# Load all feather files
path = '/lhome/nicholas/sims_kul21/jitterNone/000000001/' 
lc = PhotometryFile(path, mode="multi")

In [ ]:
# Unpack all zip files in the path folder
lc.unpack()

In [ ]:
# Merge all observations for the same quarter 
lc1, ncam = lc.merge(quarter=1)
ncam

In [ ]:
binned = lc1.bin()
binned.head()

In [ ]:
# We can also plot the binned data directly along the merged light curve
fig, ax = lc1.plot(time_unit="h", binsize=1, merged=ncam);